# Stage 5: Football Match Outcome Prediction

## Objective

The goal of this stage is to build a machine learning model that predicts the outcome of a football match:

- `home_win`
- `away_win`
- `draw`

This stage extends the earlier SQL, Python, and Power BI work into a complete machine learning workflow.

## Project Workflow

The prediction pipeline follows:

**BigQuery → SQL feature engineering → Python/Pandas → Machine Learning → Model Evaluation**

## Data

The project uses football data stored in the BigQuery project:

`football-analytics-507017`

Dataset:

`football_data`

The main source tables used in this stage include:

- `games`
- `player_valuations`
- `club games`

Additional historical information is derived from these tables using SQL views.

## Prediction Features

The model uses information that should be available before a match is played.

The main features are:

- Historical average player valuation of the home club
- Historical average player valuation of the away club
- Historical win rate of the home club
- Historical win rate of the away club
- Recent points earned by the home club
- Recent points earned by the away club

Recent form is based on the previous five available matches for each club.

## Important Methodological Consideration

A major focus of this stage is preventing **temporal data leakage**.

A football match should only be predicted using information that would have been available before that match.

The project therefore evolved from an initial random train/test split to a leakage-safe chronological evaluation.

The final evaluation uses:

- Training data: 2012-08-09 to 2024-10-23
- Test data: 2024-10-24 to 2026-07-06

This provides a more realistic simulation of predicting future matches from historical data.

## Models Tested

The following approaches are evaluated during this stage:

1. Logistic Regression
2. Scaled Logistic Regression
3. Balanced Logistic Regression
4. Logistic Regression with recent-form features
5. Matchup-difference features
6. Chronological Logistic Regression
7. Balanced Chronological Logistic Regression
8. Random Forest

## Evaluation Metrics

Because football match outcomes contain three classes and draws are harder to predict, accuracy alone is not sufficient.

The main evaluation metrics are:

- Accuracy
- Macro F1
- Precision
- Recall
- Confusion Matrix
- Draw recall

Macro F1 is particularly useful because it gives equal importance to all three outcome classes.

## Final Goal

The objective is not to produce a perfect football prediction system.

Instead, this stage demonstrates a practical machine learning workflow involving:

- Target definition
- SQL-based feature engineering
- Data preparation
- Temporal leakage detection
- Chronological train/test splitting
- Class imbalance handling
- Model comparison
- Evaluation
- Interpretation
- Documentation of limitations

In [1]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="football-analytics-507017")

In [2]:
query = """
SELECT *
FROM `football-analytics-507017.football_data.model_training_data`
"""

df = client.query(query).to_dataframe()

df.head()

c:\Users\Aniket\AppData\Local\Programs\Python\Python314\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,game_id,home_club_id,away_club_id,match_result,home_avg_valuation,home_win_rate,away_avg_valuation,away_win_rate
0,4375253,14,4441,home_win,816179.435484,0.426087,8.228801e+05,0.400000
1,4375266,14,413,home_win,816179.435484,0.426087,1.128884e+06,0.500000
2,4375276,14,122,draw,816179.435484,0.426087,1.554291e+06,0.383333
3,4375294,14,316,home_win,816179.435484,0.426087,3.486631e+05,0.237288
4,4375306,14,2446,home_win,816179.435484,0.426087,4.059322e+05,0.288136


In [4]:
df.shape

(83400, 8)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 83400 entries, 0 to 83399
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   game_id             83400 non-null  Int64  
 1   home_club_id        83400 non-null  Int64  
 2   away_club_id        83400 non-null  Int64  
 3   match_result        83400 non-null  str    
 4   home_avg_valuation  83400 non-null  float64
 5   home_win_rate       83400 non-null  float64
 6   away_avg_valuation  83400 non-null  float64
 7   away_win_rate       83400 non-null  float64
dtypes: Int64(3), float64(4), str(1)
memory usage: 5.9 MB


In [6]:
X = df[
    [
        "home_avg_valuation",
        "home_win_rate",
        "away_avg_valuation",
        "away_win_rate"
    ]
]

y = df["match_result"]

X.head()

,home_avg_valuation,home_win_rate,away_avg_valuation,away_win_rate
0,816179.435484,0.426087,8.228801e+05,0.400000
1,816179.435484,0.426087,1.128884e+06,0.500000
2,816179.435484,0.426087,1.554291e+06,0.383333
3,816179.435484,0.426087,3.486631e+05,0.237288
4,816179.435484,0.426087,4.059322e+05,0.288136


In [7]:
y.value_counts()

match_result
home_win    37783
away_win    26687
draw        18930
Name: count, dtype: int64

In [8]:
y_encoded = y.map({
    "home_win": 0,
    "away_win": 1,
    "draw": 2
})

y_encoded.value_counts()

match_result
0    37783
1    26687
2    18930
Name: count, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [11]:
print("Training data:", X_train.shape)
print("Test data:", X_test.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

Training data: (66720, 4)
Test data: (16680, 4)
Training target: (66720,)
Test target: (16680,)


In [12]:
from sklearn.linear_model import LogisticRegression

In [13]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [14]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [15]:
y_pred = model.predict(X_test)

In [16]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 53.53%


In [17]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["home_win", "away_win", "draw"]
))

              precision    recall  f1-score   support

    home_win       0.55      0.80      0.65      7557
    away_win       0.50      0.54      0.52      5337
        draw       0.00      0.00      0.00      3786

    accuracy                           0.54     16680
   macro avg       0.35      0.45      0.39     16680
weighted avg       0.41      0.54      0.46     16680



c:\Users\Aniket\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Aniket\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Aniket\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [18]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[6054 1503    0]
 [2462 2875    0]
 [2460 1326    0]]


In [19]:
import pandas as pd

pd.Series(y_pred).value_counts()

0    10976
1     5704
Name: count, dtype: int64

In [20]:
coefficients = pd.DataFrame(
    model.coef_,
    columns=X.columns,
    index=["home_win", "away_win", "draw"]
)

coefficients

,home_avg_valuation,home_win_rate,away_avg_valuation,away_win_rate
home_win,9.459620e-08,8.068105e-14,-4.876082e-08,4.162065e-14
away_win,-7.846184e-08,-2.111308e-14,6.356449e-08,1.281698e-14
draw,-1.613435e-08,-5.956796e-14,-1.480367e-08,-5.443763e-14


In [21]:
from sklearn.preprocessing import StandardScaler

In [22]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [23]:
scaled_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

scaled_model.fit(X_train_scaled, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [24]:
y_pred_scaled = scaled_model.predict(X_test_scaled)

In [25]:
accuracy_scaled = accuracy_score(y_test, y_pred_scaled)

print(f"Scaled Accuracy: {accuracy_scaled:.2%}")

Scaled Accuracy: 53.86%


In [26]:
pd.Series(y_pred_scaled).value_counts()

0    11289
1     5390
2        1
Name: count, dtype: int64

In [27]:
print(classification_report(
    y_test,
    y_pred_scaled,
    target_names=["home_win", "away_win", "draw"]
))

              precision    recall  f1-score   support

    home_win       0.55      0.82      0.66      7557
    away_win       0.51      0.52      0.52      5337
        draw       0.00      0.00      0.00      3786

    accuracy                           0.54     16680
   macro avg       0.35      0.45      0.39     16680
weighted avg       0.41      0.54      0.46     16680



In [28]:
balanced_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

In [29]:
balanced_model.fit(X_train_scaled, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [30]:
y_pred_balanced = balanced_model.predict(X_test_scaled)

In [31]:
pd.Series(y_pred_balanced).value_counts()

0    6353
1    5653
2    4674
Name: count, dtype: int64

In [32]:
accuracy_balanced = accuracy_score(y_test, y_pred_balanced)

print(f"Balanced Accuracy: {accuracy_balanced:.2%}")

Balanced Accuracy: 48.81%


In [33]:
print(classification_report(
    y_test,
    y_pred_balanced,
    target_names=["home_win", "away_win", "draw"]
))

              precision    recall  f1-score   support

    home_win       0.64      0.53      0.58      7557
    away_win       0.50      0.53      0.52      5337
        draw       0.27      0.33      0.30      3786

    accuracy                           0.49     16680
   macro avg       0.47      0.47      0.47     16680
weighted avg       0.51      0.49      0.50     16680



In [34]:
from sklearn.ensemble import RandomForestClassifier

In [35]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [36]:
rf_model.fit(X_train, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

In [37]:
y_pred_rf = rf_model.predict(X_test)

In [38]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest Accuracy: {accuracy_rf:.2%}")

Random Forest Accuracy: 48.21%


In [39]:
pd.Series(y_pred_rf).value_counts()

0    8811
1    5338
2    2531
Name: count, dtype: int64

In [40]:
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["home_win", "away_win", "draw"]
))

              precision    recall  f1-score   support

    home_win       0.55      0.65      0.60      7557
    away_win       0.47      0.47      0.47      5337
        draw       0.26      0.18      0.21      3786

    accuracy                           0.48     16680
   macro avg       0.43      0.43      0.42     16680
weighted avg       0.46      0.48      0.47     16680



## Stage 5: Baseline Model Results

### Dataset
- 83,400 matches
- 4 input features
- 3 target classes: home_win, away_win, draw
- Train/test split: 80/20
- Stratified split with random_state=42

### Models Tested

| Model | Accuracy | Key Observation |
|---|---:|---|
| Logistic Regression | 53.53% | Did not predict draws |
| Scaled Logistic Regression | 53.86% | Slight improvement, still almost no draws |
| Balanced Logistic Regression | 48.81% | Predicted all three outcomes, better class balance |
| Random Forest | 48.21% | Worse overall and weak draw detection |

### Key Findings

- Feature scaling provided only a small improvement.
- Class weighting substantially improved draw detection but reduced overall accuracy.
- Random Forest performed worse than Logistic Regression with the current features.
- Accuracy alone is not sufficient for evaluating this three-class problem.
- The current feature set is limited to club valuation and historical win rate.
- The current features were calculated using the full historical dataset, so temporal data leakage may be present.
- A future improvement will be to create time-aware features using only information available before each match.

In [41]:
query = """
SELECT *
FROM `football-analytics-507017.football_data.model_training_data_final`
"""

df_form = client.query(query).to_dataframe()

df_form.shape

c:\Users\Aniket\AppData\Local\Programs\Python\Python314\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(78023, 13)

In [42]:
df_form.info()

<class 'pandas.DataFrame'>
RangeIndex: 78023 entries, 0 to 78022
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   game_id              78023 non-null  Int64  
 1   date                 78023 non-null  dbdate 
 2   home_club_id         78023 non-null  Int64  
 3   away_club_id         78023 non-null  Int64  
 4   match_result         78023 non-null  str    
 5   home_avg_valuation   78023 non-null  float64
 6   home_win_rate        78023 non-null  float64
 7   away_avg_valuation   78023 non-null  float64
 8   away_win_rate        78023 non-null  float64
 9   home_recent_points   78023 non-null  Int64  
 10  home_recent_matches  78023 non-null  Int64  
 11  away_recent_points   78023 non-null  Int64  
 12  away_recent_matches  78023 non-null  Int64  
dtypes: Int64(7), dbdate(1), float64(4), str(1)
memory usage: 8.8 MB


In [43]:
df_form.head()

,game_id,date,home_club_id,away_club_id,match_result,home_avg_valuation,home_win_rate,away_avg_valuation,away_win_rate,home_recent_points,home_recent_matches,away_recent_points,away_recent_matches
0,4375253,2024-08-11,14,4441,home_win,816179.435484,0.426087,8.228801e+05,0.400000,4,5,10,5
1,4375266,2024-08-25,14,413,home_win,816179.435484,0.426087,1.128884e+06,0.500000,5,5,1,5
2,4375276,2024-09-25,14,122,draw,816179.435484,0.426087,1.554291e+06,0.383333,8,5,7,5
3,4375306,2024-10-26,14,2446,home_win,816179.435484,0.426087,4.059322e+05,0.288136,11,5,7,5
4,4375324,2024-11-24,14,4467,home_win,816179.435484,0.426087,4.225510e+05,0.305085,15,5,5,5


In [44]:
X_form = df_form[
    [
        "home_avg_valuation",
        "home_win_rate",
        "away_avg_valuation",
        "away_win_rate",
        "home_recent_points",
        "away_recent_points"
    ]
]

y_form = df_form["match_result"].map({
    "home_win": 0,
    "away_win": 1,
    "draw": 2
})

X_form.shape, y_form.value_counts()

((78023, 6),
 match_result
 0    35306
 1    24668
 2    18049
 Name: count, dtype: int64)

In [45]:
from sklearn.model_selection import train_test_split

X_train_form, X_test_form, y_train_form, y_test_form = train_test_split(
    X_form,
    y_form,
    test_size=0.2,
    random_state=42,
    stratify=y_form
)

X_train_form.shape, X_test_form.shape, y_train_form.shape, y_test_form.shape

((62418, 6), (15605, 6), (62418,), (15605,))

In [46]:
from sklearn.preprocessing import StandardScaler

scaler_form = StandardScaler()

X_train_form_scaled = scaler_form.fit_transform(X_train_form)
X_test_form_scaled = scaler_form.transform(X_test_form)

In [47]:
X_train_form_scaled.shape, X_test_form_scaled.shape

((62418, 6), (15605, 6))

In [48]:
from sklearn.linear_model import LogisticRegression

form_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

form_model.fit(X_train_form_scaled, y_train_form)

y_pred_form = form_model.predict(X_test_form_scaled)

In [49]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test_form, y_pred_form))

print("\nClassification Report:")
print(
    classification_report(
        y_test_form,
        y_pred_form,
        target_names=["home_win", "away_win", "draw"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_form, y_pred_form))

Accuracy: 0.5331624479333547

Classification Report:
              precision    recall  f1-score   support

    home_win       0.54      0.82      0.65      7061
    away_win       0.51      0.51      0.51      4934
        draw       1.00      0.00      0.00      3610

    accuracy                           0.53     15605
   macro avg       0.69      0.44      0.39     15605
weighted avg       0.64      0.53      0.46     15605


Confusion Matrix:
[[5816 1245    0]
 [2431 2503    0]
 [2485 1124    1]]


In [50]:
balanced_form_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

balanced_form_model.fit(
    X_train_form_scaled,
    y_train_form
)

y_pred_balanced_form = balanced_form_model.predict(
    X_test_form_scaled
)

In [51]:
print(
    "Accuracy:",
    accuracy_score(y_test_form, y_pred_balanced_form)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_form,
        y_pred_balanced_form,
        target_names=["home_win", "away_win", "draw"]
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_form,
        y_pred_balanced_form
    )
)

Accuracy: 0.48958667093880165

Classification Report:
              precision    recall  f1-score   support

    home_win       0.63      0.55      0.59      7061
    away_win       0.50      0.51      0.51      4934
        draw       0.28      0.34      0.31      3610

    accuracy                           0.49     15605
   macro avg       0.47      0.47      0.47     15605
weighted avg       0.51      0.49      0.50     15605


Confusion Matrix:
[[3876 1385 1800]
 [1050 2528 1356]
 [1217 1157 1236]]


## Stage 5: Model v2 - Recent Form

### Feature Engineering

Added recent form using each club's previous 5 matches before the target match.

Features added:
- `home_recent_points`
- `away_recent_points`

Each feature ranges from 0 to 15 points.

Only matches before the target match date were used. Same-day matches were excluded because exact kickoff times were unavailable.

Matches where either club had fewer than 5 previous matches were removed.

Final dataset:
- 78,023 matches
- 6 model features

### Model Features

The model used:

1. `home_avg_valuation`
2. `home_win_rate`
3. `away_avg_valuation`
4. `away_win_rate`
5. `home_recent_points`
6. `away_recent_points`

The data was split 80/20 using stratification with `random_state=42`.

Features were standardized using `StandardScaler`, fitted only on the training set.

### Results

| Model | Accuracy | Macro F1 | Draw Recall |
|---|---:|---:|---:|
| Logistic Regression | 53.32% | 0.39 | 0.00 |
| Balanced Logistic Regression | 48.96% | 0.47 | 0.34 |

### Logistic Regression

The model achieved 53.32% accuracy but almost completely ignored the draw class.

Confusion matrix:

[[5816, 1245, 0],
 [2431, 2503, 0],
 [2485, 1124, 1]]

### Balanced Logistic Regression

Using `class_weight="balanced"` reduced overall accuracy to 48.96%, but improved performance on the minority draw class.

Confusion matrix:

[[3876, 1385, 1800],
 [1050, 2528, 1356],
 [1217, 1157, 1236]]

Draw recall improved from 0.00 to 0.34.

Macro F1 improved from 0.39 to 0.47.

### Key Findings

- Adding recent form did not meaningfully improve standard Logistic Regression.
- The standard model still struggled to predict draws.
- Class weighting substantially improved draw detection.
- Balanced Logistic Regression had better macro F1 but lower overall accuracy.
- The current feature set was still affected by historical feature leakage.

### Conclusion

Recent form alone was not enough to substantially improve the model.

The next step is to create leakage-safe historical valuation and historical win-rate features, followed by a chronological train/test split to simulate predicting future matches.

In [52]:
query = """
SELECT *
FROM `football-analytics-507017.football_data.model_training_data_time_aware`
"""

df_time = client.query(query).to_dataframe()

df_time.shape

c:\Users\Aniket\AppData\Local\Programs\Python\Python314\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(77137, 13)

In [53]:
df_time.info()

<class 'pandas.DataFrame'>
RangeIndex: 77137 entries, 0 to 77136
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   game_id                        77137 non-null  Int64  
 1   date                           77137 non-null  dbdate 
 2   home_club_id                   77137 non-null  Int64  
 3   away_club_id                   77137 non-null  Int64  
 4   match_result                   77137 non-null  str    
 5   home_historical_avg_valuation  77137 non-null  float64
 6   away_historical_avg_valuation  77137 non-null  float64
 7   home_historical_win_rate       77137 non-null  float64
 8   away_historical_win_rate       77137 non-null  float64
 9   home_recent_points             77137 non-null  Int64  
 10  home_recent_matches            77137 non-null  Int64  
 11  away_recent_points             77137 non-null  Int64  
 12  away_recent_matches            77137 non-null  Int64  
dt

In [54]:
df_time = df_time.sort_values("date").reset_index(drop=True)

split_index = int(len(df_time) * 0.8)

train_df = df_time.iloc[:split_index]
test_df = df_time.iloc[split_index:]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("Train dates:", train_df["date"].min(), "to", train_df["date"].max())
print("Test dates:", test_df["date"].min(), "to", test_df["date"].max())

Train: (61709, 13)
Test: (15428, 13)
Train dates: 2012-08-09 to 2024-10-24
Test dates: 2024-10-24 to 2026-07-06


In [55]:
split_date = train_df["date"].max()

train_df = df_time[df_time["date"] < split_date]
test_df = df_time[df_time["date"] >= split_date]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("Train dates:", train_df["date"].min(), "to", train_df["date"].max())
print("Test dates:", test_df["date"].min(), "to", test_df["date"].max())

Train: (61681, 13)
Test: (15456, 13)
Train dates: 2012-08-09 to 2024-10-23
Test dates: 2024-10-24 to 2026-07-06


In [56]:
features = [
    "home_historical_avg_valuation",
    "away_historical_avg_valuation",
    "home_historical_win_rate",
    "away_historical_win_rate",
    "home_recent_points",
    "away_recent_points"
]

X_train_time = train_df[features]
X_test_time = test_df[features]

y_train_time = train_df["match_result"].map({
    "home_win": 0,
    "away_win": 1,
    "draw": 2
})

y_test_time = test_df["match_result"].map({
    "home_win": 0,
    "away_win": 1,
    "draw": 2
})

print("X_train:", X_train_time.shape)
print("X_test:", X_test_time.shape)
print("y_train:", y_train_time.shape)
print("y_test:", y_test_time.shape)

X_train: (61681, 6)
X_test: (15456, 6)
y_train: (61681,)
y_test: (15456,)


In [57]:
print("Training distribution:")
print(y_train_time.value_counts(normalize=True).sort_index())

print("\nTest distribution:")
print(y_test_time.value_counts(normalize=True).sort_index())

Training distribution:
match_result
0    0.45403
1    0.31410
2    0.23187
Name: proportion, dtype: float64

Test distribution:
match_result
0    0.444876
1    0.320264
2    0.234860
Name: proportion, dtype: float64


In [58]:
from sklearn.preprocessing import StandardScaler

scaler_time = StandardScaler()

X_train_time_scaled = scaler_time.fit_transform(X_train_time)
X_test_time_scaled = scaler_time.transform(X_test_time)

print("Scaled training shape:", X_train_time_scaled.shape)
print("Scaled test shape:", X_test_time_scaled.shape)

Scaled training shape: (61681, 6)
Scaled test shape: (15456, 6)


In [59]:
from sklearn.linear_model import LogisticRegression

time_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

time_model.fit(X_train_time_scaled, y_train_time)

y_pred_time = time_model.predict(X_test_time_scaled)

print("Predictions:", len(y_pred_time))

Predictions: 15456


In [60]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy_time = accuracy_score(y_test_time, y_pred_time)

print("Accuracy:", accuracy_time)

print("\nClassification Report:")
print(
    classification_report(
        y_test_time,
        y_pred_time,
        target_names=["home_win", "away_win", "draw"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_time, y_pred_time))

Accuracy: 0.5062758799171843

Classification Report:
              precision    recall  f1-score   support

    home_win       0.51      0.83      0.64      6876
    away_win       0.48      0.42      0.45      4950
        draw       0.00      0.00      0.00      3630

    accuracy                           0.51     15456
   macro avg       0.33      0.42      0.36     15456
weighted avg       0.38      0.51      0.43     15456


Confusion Matrix:
[[5738 1137    1]
 [2861 2087    2]
 [2550 1080    0]]


## Stage 5: Leakage-Safe Time-Aware Model

### Why the Earlier Model Was Not Enough

The initial models used a random train/test split. Although useful for learning, this can give an overly optimistic estimate for historical sports data because information from different time periods can be mixed between training and testing.

The historical valuation and win-rate features were also initially calculated using the full dataset, which introduced potential temporal leakage.

To make the evaluation more realistic, time-aware historical features were created using only information available before each target match.

### Time-Aware Features

The model now uses:

1. `home_historical_avg_valuation`
2. `away_historical_avg_valuation`
3. `home_historical_win_rate`
4. `away_historical_win_rate`
5. `home_recent_points`
6. `away_recent_points`

Recent form uses each club's previous 5 matches.

Historical win rate uses only matches before the target match date.

Historical valuation uses the latest available valuation records on or before the target match date.

### Data Quality

Final leakage-safe dataset:

- 77,137 matches
- 6 predictive features
- No missing values in the required features

### Chronological Train/Test Split

Instead of randomly splitting the data, matches were ordered by date.

Training period:
- 2012-08-09 to 2024-10-23
- 61,681 matches

Test period:
- 2024-10-24 to 2026-07-06
- 15,456 matches

There is no date overlap between the training and test periods.

The chronological split better represents a real prediction scenario because the model is trained on past matches and evaluated on future matches.

### Class Distribution

Training:
- Home win: 45.40%
- Away win: 31.41%
- Draw: 23.19%

Test:
- Home win: 44.49%
- Away win: 32.03%
- Draw: 23.49%

The class distributions were reasonably similar between training and test data.

### Leakage-Safe Logistic Regression

Features were standardized using `StandardScaler`.

The scaler was fitted only on the training data and then applied to the test data.

The chronological Logistic Regression model achieved:

- Accuracy: 50.63%
- Macro F1: 0.36
- Home-win recall: 0.83
- Away-win recall: 0.42
- Draw recall: 0.00

Confusion matrix:

[[5738, 1137, 1],
 [2861, 2087, 2],
 [2550, 1080, 0]]

### Comparison With Earlier Models

| Model | Split | Accuracy | Macro F1 | Draw Recall |
|---|---|---:|---:|---:|
| Scaled Logistic Regression | Random | 53.86% | 0.39 | 0.00 |
| Logistic Regression + Recent Form | Random | 53.32% | 0.39 | 0.00 |
| Leakage-Safe Logistic Regression | Chronological | 50.63% | 0.36 | 0.00 |

### Key Findings

- Chronological evaluation produced lower accuracy than the earlier random split.
- This suggests the earlier results were somewhat optimistic.
- The model still strongly favors predicting home wins.
- Draw prediction remains the biggest weakness.
- The model is now being evaluated in a more realistic future-prediction setting.

### Conclusion

The chronological evaluation provides a more honest estimate of model performance.

The next experiment will focus on feature engineering by creating matchup-difference features:

- `valuation_diff`
- `win_rate_diff`
- `recent_points_diff`

These features will represent the relative strength of the two teams rather than only their individual values.

In [61]:
X_train_diff = X_train_time.copy()
X_test_diff = X_test_time.copy()

X_train_diff["valuation_diff"] = (
    X_train_diff["home_historical_avg_valuation"]
    - X_train_diff["away_historical_avg_valuation"]
)

X_test_diff["valuation_diff"] = (
    X_test_diff["home_historical_avg_valuation"]
    - X_test_diff["away_historical_avg_valuation"]
)

X_train_diff["win_rate_diff"] = (
    X_train_diff["home_historical_win_rate"]
    - X_train_diff["away_historical_win_rate"]
)

X_test_diff["win_rate_diff"] = (
    X_test_diff["home_historical_win_rate"]
    - X_test_diff["away_historical_win_rate"]
)

X_train_diff["recent_points_diff"] = (
    X_train_diff["home_recent_points"]
    - X_train_diff["away_recent_points"]
)

X_test_diff["recent_points_diff"] = (
    X_test_diff["home_recent_points"]
    - X_test_diff["away_recent_points"]
)

print("Training shape:", X_train_diff.shape)
print("Test shape:", X_test_diff.shape)

Training shape: (61681, 9)
Test shape: (15456, 9)


In [62]:
from sklearn.preprocessing import StandardScaler

scaler_diff = StandardScaler()

X_train_diff_scaled = scaler_diff.fit_transform(X_train_diff)
X_test_diff_scaled = scaler_diff.transform(X_test_diff)

print("Scaled training shape:", X_train_diff_scaled.shape)
print("Scaled test shape:", X_test_diff_scaled.shape)

Scaled training shape: (61681, 9)
Scaled test shape: (15456, 9)


In [63]:
from sklearn.linear_model import LogisticRegression

diff_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

diff_model.fit(X_train_diff_scaled, y_train_time)

y_pred_diff = diff_model.predict(X_test_diff_scaled)

print("Predictions:", len(y_pred_diff))

Predictions: 15456


In [64]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

accuracy_diff = accuracy_score(y_test_time, y_pred_diff)

print("Accuracy:", accuracy_diff)

print("\nClassification Report:")
print(
    classification_report(
        y_test_time,
        y_pred_diff,
        target_names=["home_win", "away_win", "draw"]
    )
)

print("\nMacro F1:", f1_score(y_test_time, y_pred_diff, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_time, y_pred_diff))

Accuracy: 0.5062758799171843

Classification Report:
              precision    recall  f1-score   support

    home_win       0.51      0.83      0.64      6876
    away_win       0.48      0.42      0.45      4950
        draw       0.00      0.00      0.00      3630

    accuracy                           0.51     15456
   macro avg       0.33      0.42      0.36     15456
weighted avg       0.38      0.51      0.43     15456


Macro F1: 0.3625642251706513

Confusion Matrix:
[[5738 1137    1]
 [2861 2087    2]
 [2548 1082    0]]


## Stage 5: Matchup Difference Features

### Feature Engineering

Three additional features were created to represent the relative strength between the home and away teams:

- `valuation_diff`
- `win_rate_diff`
- `recent_points_diff`

Each difference was calculated as:

`home value - away value`

These features were added to the existing six time-aware features.

### Results

| Model | Accuracy | Macro F1 | Draw Recall |
|---|---:|---:|---:|
| Leakage-Safe Logistic Regression | 50.63% | 0.36 | 0.00 |
| + Matchup Difference Features | 50.63% | 0.36 | 0.00 |

### Key Finding

Adding matchup-difference features did not improve the chronological Logistic Regression model.

The new features were derived from information that was already available to the model through the individual home and away features, so they did not provide substantial additional predictive information.

### Conclusion

The model's main limitation is not simply how the existing features are represented.

The model continues to strongly favor home wins and does not meaningfully predict draws.

The next experiment will use class weighting with the chronological model to determine whether improving minority-class treatment can improve overall class balance.

In [65]:
balanced_time_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

balanced_time_model.fit(
    X_train_time_scaled,
    y_train_time
)

y_pred_balanced_time = balanced_time_model.predict(
    X_test_time_scaled
)

print("Predictions:", len(y_pred_balanced_time))

Predictions: 15456


In [66]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

accuracy_balanced_time = accuracy_score(
    y_test_time,
    y_pred_balanced_time
)

print("Accuracy:", accuracy_balanced_time)

print("\nClassification Report:")
print(
    classification_report(
        y_test_time,
        y_pred_balanced_time,
        target_names=["home_win", "away_win", "draw"]
    )
)

print("\nMacro F1:", f1_score(
    y_test_time,
    y_pred_balanced_time,
    average="macro"
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_time,
    y_pred_balanced_time
))

Accuracy: 0.46053312629399584

Classification Report:
              precision    recall  f1-score   support

    home_win       0.59      0.53      0.56      6876
    away_win       0.47      0.48      0.47      4950
        draw       0.26      0.30      0.28      3630

    accuracy                           0.46     15456
   macro avg       0.44      0.44      0.44     15456
weighted avg       0.47      0.46      0.47     15456


Macro F1: 0.4373452263241348

Confusion Matrix:
[[3637 1472 1767]
 [1220 2393 1337]
 [1261 1281 1088]]


## Stage 5: Balanced Chronological Model

### Class Imbalance

The standard chronological Logistic Regression model strongly favored home wins and failed to predict draws.

To address this, a second Logistic Regression model was trained using:

`class_weight="balanced"`

This automatically gives more weight to underrepresented classes during training.

### Results

| Model | Accuracy | Macro F1 | Draw Recall |
|---|---:|---:|---:|
| Chronological Logistic Regression | 50.63% | 0.36 | 0.00 |
| Balanced Chronological Logistic Regression | 46.05% | 0.44 | 0.30 |

### Classification Report

The balanced model achieved:

- Home-win precision: 0.59
- Home-win recall: 0.53
- Away-win precision: 0.47
- Away-win recall: 0.48
- Draw precision: 0.26
- Draw recall: 0.30
- Macro F1: 0.44

### Confusion Matrix

[[3637, 1472, 1767],
 [1220, 2393, 1337],
 [1261, 1281, 1088]]

### Key Findings

- Class weighting substantially improved the balance between outcome classes.
- Draw recall improved from 0.00 to 0.30.
- Macro F1 improved from 0.36 to 0.44.
- Overall accuracy decreased from 50.63% to 46.05%.
- The model no longer heavily favors home wins.

### Conclusion

The balanced model provides a more balanced three-class prediction than the standard chronological model.

Although its overall accuracy is lower, its higher macro F1 indicates better performance across all three outcome classes.

For this three-class problem, macro F1 is therefore more informative than accuracy alone.

The balanced chronological model is currently the strongest Logistic Regression model tested under the leakage-safe evaluation setup.

In [67]:
from sklearn.ensemble import RandomForestClassifier

rf_time_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_time_model.fit(
    X_train_time,
    y_train_time
)

y_pred_rf_time = rf_time_model.predict(
    X_test_time
)

print("Predictions:", len(y_pred_rf_time))

Predictions: 15456


In [68]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

accuracy_rf_time = accuracy_score(
    y_test_time,
    y_pred_rf_time
)

print("Accuracy:", accuracy_rf_time)

print("\nClassification Report:")
print(
    classification_report(
        y_test_time,
        y_pred_rf_time,
        target_names=["home_win", "away_win", "draw"]
    )
)

print("\nMacro F1:", f1_score(
    y_test_time,
    y_pred_rf_time,
    average="macro"
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_time,
    y_pred_rf_time
))

Accuracy: 0.4920419254658385

Classification Report:
              precision    recall  f1-score   support

    home_win       0.53      0.75      0.62      6876
    away_win       0.47      0.44      0.45      4950
        draw       0.27      0.09      0.13      3630

    accuracy                           0.49     15456
   macro avg       0.42      0.42      0.40     15456
weighted avg       0.45      0.49      0.45     15456


Macro F1: 0.4021100390140284

Confusion Matrix:
[[5124 1322  430]
 [2356 2156  438]
 [2224 1081  325]]


## Stage 5: Random Forest

### Model

A Random Forest classifier was trained using the same leakage-safe chronological train/test split as the Logistic Regression models.

The model used the six original time-aware features:

- `home_historical_avg_valuation`
- `away_historical_avg_valuation`
- `home_historical_win_rate`
- `away_historical_win_rate`
- `home_recent_points`
- `away_recent_points`

### Results

| Model | Accuracy | Macro F1 | Draw Recall |
|---|---:|---:|---:|
| Chronological Logistic Regression | 50.63% | 0.36 | 0.00 |
| Balanced Chronological Logistic Regression | 46.05% | 0.44 | 0.30 |
| Random Forest | 49.20% | 0.40 | 0.09 |

### Key Findings

The Random Forest achieved 49.20% accuracy and a macro F1 of 0.40.

Compared with the balanced Logistic Regression model, Random Forest had:

- Higher accuracy: 49.20% vs 46.05%
- Lower macro F1: 0.40 vs 0.44
- Much lower draw recall: 0.09 vs 0.30

The Random Forest continued to favor home wins, with a home-win recall of 0.75, while correctly identifying only 9% of draws.

### Conclusion

The Random Forest did not outperform the balanced chronological Logistic Regression model on the overall three-class evaluation.

The balanced Logistic Regression model remains the strongest model tested because it provides better balance across home wins, away wins, and draws, as reflected by its higher macro F1 and substantially better draw recall.

No further model tuning is planned for this stage.

# Stage 5: Match Outcome Prediction

## Objective

The goal of this stage was to build a machine learning model that predicts football match outcomes as:

- `home_win`
- `away_win`
- `draw`

The model used historical information available before each match, including:

- Historical average player valuation
- Historical club win rate
- Recent form from the previous five matches

## Data Preparation

A total of 88,958 matches were available in the original match dataset.

After removing matches without the required historical features and recent-form information, the final leakage-safe dataset contained 77,137 matches.

The data was split chronologically:

- Training period: 2012-08-09 to 2024-10-23
- Test period: 2024-10-24 to 2026-07-06

The chronological split was used to avoid training on information from the future.

## Feature Engineering

The final model used six time-aware features:

- `home_historical_avg_valuation`
- `away_historical_avg_valuation`
- `home_historical_win_rate`
- `away_historical_win_rate`
- `home_recent_points`
- `away_recent_points`

Additional matchup-difference features were also tested:

- `valuation_diff`
- `win_rate_diff`
- `recent_points_diff`

These did not improve model performance.

## Models Tested

Several models and evaluation strategies were compared.

| Model | Accuracy | Macro F1 | Draw Recall |
|---|---:|---:|---:|
| Scaled Logistic Regression, random split | 53.86% | 0.39 | 0.00 |
| Balanced Logistic Regression, random split | 48.81% | 0.47 | 0.33 |
| Logistic Regression + recent form, random split | 53.32% | 0.39 | 0.00 |
| Balanced LR + recent form, random split | 48.96% | 0.47 | 0.34 |
| Chronological Logistic Regression | 50.63% | 0.36 | 0.00 |
| Chronological LR + difference features | 50.63% | 0.36 | 0.00 |
| Balanced Chronological Logistic Regression | 46.05% | 0.44 | 0.30 |
| Random Forest | 49.20% | 0.40 | 0.09 |

## Final Model

The Balanced Chronological Logistic Regression model was selected as the strongest model for this three-class prediction task.

Results on the chronological test set:

- Accuracy: 46.05%
- Macro F1: 0.44
- Home-win recall: 0.53
- Away-win recall: 0.48
- Draw recall: 0.30

## Key Findings

The initial random-split experiments produced higher accuracy, but chronological evaluation showed that these results were somewhat optimistic because historical and future matches were mixed during random splitting.

The chronological evaluation provided a more realistic estimate of how the model performs on future matches.

Class weighting significantly improved the model's ability to identify draws.

The balanced model reduced overall accuracy compared with the standard chronological model, but improved macro F1 from 0.36 to 0.44 and draw recall from 0.00 to 0.30.

Random Forest achieved 49.20% accuracy, but its draw recall was only 0.09, making it less balanced than the selected Logistic Regression model.

## Limitations

The model should not be considered a production-level football prediction system.

Important limitations include:

- Historical valuation is based on available valuation snapshots rather than a perfect match-day squad valuation.
- The dataset does not contain every factor that can influence match outcomes.
- Injuries, suspensions, tactics, lineups, home advantage details, and other contextual factors were not included.
- Same-day matches were treated using available date information because exact kickoff times were not available.
- The model was evaluated on historical data and should not be interpreted as a guaranteed predictor of future results.

## Conclusion

The main learning outcome of this stage was not achieving the highest possible prediction accuracy.

The project demonstrated a complete machine learning workflow:

1. Defined a prediction target.
2. Built features using SQL.
3. Identified and addressed temporal leakage.
4. Created a chronological train/test split.
5. Compared multiple machine learning approaches.
6. Evaluated class imbalance using macro F1 and class-level recall.
7. Tested additional feature engineering.
8. Selected a final model based on the requirements of the three-class problem.
9. Documented model limitations.

The final model provides a useful baseline for future experimentation while demonstrating practical skills in SQL, Python, feature engineering, machine learning, model evaluation, and data-quality reasoning.